In [1]:
# 프로젝트 하기 위해서 필요한 것
# 1. 전처리된 데이터셋
# 2. 특정한 이미지를 판별
# ㄴ 해당 이미지가 없음 -> 직접
# ㄴ 웹 이미지를 1000장 -> 크롤링

# 크롤링을 이제 진행해볼 예정
# 우측 상단 커널 선택 -> myenv
print("123")

123


In [2]:
!pip install selenium

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ------------- -------------------------- 3.1/9.5 MB 14.1 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 26.9 MB/s  0:00:00

   ----- ---------------------------------- 1/8 [wsproto]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------------ --------- 6/8 [trio-websocket]
   ----------------------------------- ---- 7/8 [selenium]
   ----------------------------------- ---- 7/8 [selenium]
   -----------------

In [6]:
# 스크래핑 vs 크롤링
# 정적인 페이지 vs 동적인 페이지
# 크롤링 합법이 아니다
# 크롤링 할 사이트에 "/robots.txt" 해보면 됨

# 셀리니움 라이브러리 설치
from selenium import webdriver as wb
driver = wb.Chrome()
url = "https://www.naver.com"
driver.get(url)

In [ ]:
# 검색어 창에 "화곡역 맛집" 입력 후
# 엔터 혹은 검색 아이콘 누르기
# 해당 태그를 바라볼 수 있게 선택자 개념 활용
# 모든 웹 페이지에는 태그들로 존재
# 해당 태그는 다음과 같이 구성
# <태그이름 id="price" name="input">300,000</태그이름>
# 1) 태그이름으로 가져오기
# 2) name으로 가져오기
# 3) id 가져오기 <- best

# 우리 검색창 id="query"
# 1) id, 태그, 가져오기 위한 라이브러리
from selenium.webdriver.common.by import By
# 2) 키보드로 "화곡역 맛집" 입력할 키보드 관련 라이브러리
from selenium.webdriver.common.keys import Keys

# 여러 태그 가져오기 => elements
# 태그 하나 가져오기 => element
search = driver.find_element(By.ID, "query")
keyword = "화곡역 맛집"
search.send_keys(keyword)
search.send_keys(Keys.ENTER) # 이거만 해도 엔터가 쳐짐
driver.close() # 하면 꺼짐

In [5]:
driver.close()

In [8]:
# 네이버 이미지 사이트에 "로봇팔" 검색 후
# 이미지 500장 다운로드

from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time # 페이지 중간 딜레이 주기 위해 시간 관련 라이브러리
import os   # 폴더를 만들자

# keyword = input("검색어를 입력하세요")
keyword = "로봇팔" # 테스트니까 바로바로 진행

if not os.path.exists(keyword): # 키워드 이름으로 된 폴더가 없으면
    os.mkdir(keyword)           # 해당 키워드로 폴더를 만들어라

driver = wb.Chrome()
url_img = "https://search.naver.com/search.naver?where=image"
driver.get(url_img) # 원래는 하나하나 단위로 끊어서 체크해야 됨

time.sleep(0.2) # 0.2s 코드 지연
search_img = driver.find_element(By.ID, "nx_query") # 검색창 태그의 ID 가져오기
search_img.send_keys(keyword) # 검색차에 keyword 입력하기(로봇팔)
search_img.send_keys(Keys.ENTER)
time.sleep(0.2) # 엔터 친 후 이미지 로딩 기다리기 (느리면 더 줘도 됨)

In [ ]:
# 주의 할 점 1. 페이지가 이동되면 그 전까지 가지고 있었던
# 인스턴스 정보가 초기화 (search_img)

# img 태그 가져오기
imgs = driver.find_elements(By.TAG_NAME, "img")
# len(imgs) # 한 페이지 전체가 떠야 됨 (194 = 다른 이미지도 껴있음)
# <img src="이미지 주소"></img>
# 이미지를 다운로드 받기 위해
# 이미지 경로가 필요
# 단, data:로 시작하면 -> 인코딩
# data: 가 아닌 친구들만 가져오도록 합시다

import requests
count = 0 # 이미지 번호를 위해 선언

for img in imgs:
    img_path = img.get_attribute("src")
    if img_path.startswith("data:"):
        continue # 반복문 이번 회차 진행 ㄴㄴ, 다음 회차로 ㄱㄱ
    response = requests.get(img_path)
    # get 방식으로 요청할 때, 응답 포맷은 3가지
    # 1) text, 2) json, 3) byte(content)
    data = response.content
    with open(f"{keyword}/{keyword}{count}.jpg", "wb") as f: # wb(write byte)
        f.write(data)
        print(f"{keyword}{count}.jpg 저장 완료")
        time.sleep(0.1)
    count += 1

In [ ]:
# 로딩 전이라 약 60장이 나온거고, 그 아래까지 뽑아내려면
# end 키를 여러번 입력하는 과정을 추가하거나 스크롤 다운 하는

from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time # 페이지 중간 딜레이 주기 위해 시간 관련 라이브러리
import os   # 폴더를 만들자
import requests

keyword = input("검색어를 입력하세요")

if not os.path.exists(keyword): # 키워드 이름으로 된 폴더가 없으면
    os.mkdir(keyword)           # 해당 키워드로 폴더를 만들어라

driver = wb.Chrome()
url_img = "https://search.naver.com/search.naver?where=image"
driver.get(url_img) # 원래는 하나하나 단위로 끊어서 체크해야 됨

time.sleep(0.2) # 0.2s 코드 지연
search_img = driver.find_element(By.ID, "nx_query") # 검색창 태그의 ID 가져오기
search_img.send_keys(keyword) # 검색차에 keyword 입력하기(로봇팔)
search_img.send_keys(Keys.ENTER)
time.sleep(0.2) # 엔터 친 후 이미지 로딩 기다리기 (느리면 더 줘도 됨)

# END 키를 20번 정도 누르자 -> 맨 땅 클릭 후 End
body = driver.find_element(By.TAG_NAME, "body")
for _ in range(20):
    body.send_keys(Keys.END)
    time.sleep(0.1)

imgs = driver.find_elements(By.TAG_NAME, "img")
count = 0 # 이미지 번호를 위해 선언

for img in imgs:
    img_path = img.get_attribute("src")
    if img_path.startswith("data:"):
        continue # 반복문 이번 회차 진행 ㄴㄴ, 다음 회차로 ㄱㄱ
    response = requests.get(img_path)
    data = response.content
    with open(f"{keyword}/{keyword}{count}.jpg", "wb") as f: # wb(write byte)
        f.write(data)
        print(f"{keyword}{count}.jpg 저장 완료")
        time.sleep(0.1)
    count += 1
driver.close()

In [ ]:
# 선택자 개념
driver.find_element(By.CSS_SELECTOR, "선택자")

# 태그는 부모와 자식
# html 태그안의 body를 바라보기 위해선
# ">" 자식 선택자
# html>body
# html>body>div>div>...>img

# " " 자손 선택자
# html img      : html 자손 중 img태그가 있으면 해당 img 태그 선택
# div img       : div 자손 중 img태그가 있으면 해당 img 태그 선택

# "~" 후행 선택자
# div~img : div 뒤에 오는 태그 중 첫 번째로 오는 img 태그
# "+" 근접 후행 선택자
# div+img : div 바로 뒤에 오는 img 태그

# 복합적으로
# div 태그 자손 중 class 이름이 abc인 img 태그 가져오기
# div img.abc       -> class 는 "." 으로 구분

# CSS 선택자 관련 미니게임 사이트
# https://flukeout.github.io/